In [0]:
# ============================================================
# CREDIT CARD DEFAULT PREDICTION — BUSINESS STRATEGY
# ============================================================

%pip install xgboost


# ------------------------------------------------------------
# 1. LOAD AND PREPARE DATA
# ------------------------------------------------------------

import pandas as pd
import numpy as np

df = spark.table("CCC_FEATURES")
pdf = df.toPandas()

# Create indicator for customers without a positive average balance
pdf["NO_POSITIVE_BALANCE"] = (
    pdf["AVG_BILL_AMT"] <= 0
).astype(int)

# Replace undefined payment-to-balance ratios with 0
pdf["PAYMENT_TO_BALANCE"] = (
    pdf["PAYMENT_TO_BALANCE"].fillna(0)
)


# ------------------------------------------------------------
# 2. DEFINE TARGET AND PREDICTORS
# ------------------------------------------------------------

features = [
    "LIMIT_BAL",
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "AGE",

    "PAY_1",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6",

    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6",

    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6",

    "AVG_BILL_AMT",
    "AVG_PAY_AMT",
    "AVG_UTILIZATION",
    "PAYMENT_TO_BALANCE",
    "NO_POSITIVE_BALANCE",
    "DELINQUENT_MONTHS",
    "MAX_DELINQUENCY",
    "BALANCE_CHANGE"
]

X = pdf[features]
y = pdf["dpnm"]


# ------------------------------------------------------------
# 3. RECREATE TRAIN/TEST SPLIT
# ------------------------------------------------------------

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("Total defaults in test set:", y_test.sum())


# ------------------------------------------------------------
# 4. TRAIN OPTIMIZED XGBOOST MODEL
# ------------------------------------------------------------

from xgboost import XGBClassifier

best_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,

    # Optimized hyperparameters from Notebook 05
    subsample=0.8,
    n_estimators=100,
    min_child_weight=1,
    max_depth=5,
    learning_rate=0.05,
    gamma=0.1,
    colsample_bytree=0.7
)

best_xgb.fit(X_train, y_train)

print("\nOptimized XGBoost model successfully trained.")


# ------------------------------------------------------------
# 5. GENERATE CUSTOMER RISK SCORES
# ------------------------------------------------------------

risk_probabilities = best_xgb.predict_proba(X_test)[:, 1]

strategy = X_test.copy()

strategy["ACTUAL_DEFAULT"] = y_test.values
strategy["PREDICTED_DEFAULT_PROBABILITY"] = risk_probabilities

# Rank customers from highest to lowest predicted risk
strategy = strategy.sort_values(
    "PREDICTED_DEFAULT_PROBABILITY",
    ascending=False
).reset_index(drop=True)

print("\nHighest-risk customers:")
display(
    strategy[
        [
            "PREDICTED_DEFAULT_PROBABILITY",
            "ACTUAL_DEFAULT",
            "PAY_1",
            "DELINQUENT_MONTHS",
            "MAX_DELINQUENCY",
            "AVG_UTILIZATION"
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 6. 10% INTERVENTION STRATEGY
# ------------------------------------------------------------

intervention_rate = 0.10

customers_to_target = int(
    len(strategy) * intervention_rate
)

targeted = strategy.head(customers_to_target)

defaults_targeted = int(
    targeted["ACTUAL_DEFAULT"].sum()
)

total_defaults = int(
    strategy["ACTUAL_DEFAULT"].sum()
)

capture_rate = (
    defaults_targeted / total_defaults
)

target_precision = (
    defaults_targeted / customers_to_target
)

# If customers were selected randomly, 10% of the
# test-set defaults would be expected to fall in that group.
random_expected_defaults = (
    total_defaults * intervention_rate
)

lift = (
    defaults_targeted / random_expected_defaults
)

print("\n10% Intervention Strategy")
print("------------------------------------")

print(
    "Customers targeted:",
    customers_to_target
)

print(
    "Defaults identified:",
    defaults_targeted
)

print(
    "Total defaults in test set:",
    total_defaults
)

print(
    f"Default capture rate: {capture_rate:.2%}"
)

print(
    f"Default rate among targeted customers: "
    f"{target_precision:.2%}"
)

print(
    "Expected defaults from random targeting:",
    round(random_expected_defaults, 1)
)

print(
    f"Lift over random targeting: {lift:.2f}x"
)


# ------------------------------------------------------------
# 7. COMPARE MULTIPLE INTERVENTION CAPACITIES
# ------------------------------------------------------------

capacity_results = []

for rate in [0.05, 0.10, 0.15, 0.20, 0.25]:

    n_target = int(
        len(strategy) * rate
    )

    targeted_group = strategy.head(n_target)

    defaults_found = int(
        targeted_group["ACTUAL_DEFAULT"].sum()
    )

    capture = (
        defaults_found / total_defaults
    )

    targeted_default_rate = (
        defaults_found / n_target
    )

    random_expected = (
        total_defaults * rate
    )

    lift = (
        defaults_found / random_expected
    )

    capacity_results.append({
        "Intervention Rate": f"{rate:.0%}",
        "Customers Targeted": n_target,
        "Defaults Identified": defaults_found,
        "Default Capture Rate": capture,
        "Targeted Default Rate": targeted_default_rate,
        "Lift vs Random": lift
    })


capacity_df = pd.DataFrame(capacity_results)

print("\nIntervention Capacity Comparison")

display(capacity_df)


# ------------------------------------------------------------
# 8. CREATE A SIMPLE BUSINESS-SUMMARY TABLE
# ------------------------------------------------------------

summary_df = capacity_df.copy()

summary_df["Default Capture Rate"] = (
    summary_df["Default Capture Rate"] * 100
).round(1).astype(str) + "%"

summary_df["Targeted Default Rate"] = (
    summary_df["Targeted Default Rate"] * 100
).round(1).astype(str) + "%"

summary_df["Lift vs Random"] = (
    summary_df["Lift vs Random"]
).round(2).astype(str) + "x"

print("\nBusiness Strategy Summary")

display(summary_df)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 56.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 116.4 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Training observations: 24000
Testing observations: 6000
Total defaults in test set: 1327

Optimized XGBoost model successfully trained.

Highest-risk customers:


PREDICTED_DEFAULT_PROBABILITY,ACTUAL_DEFAULT,PAY_1,DELINQUENT_MONTHS,MAX_DELINQUENCY,AVG_UTILIZATION
0.8807164,1,2,6,2,0.011363636363636364
0.8732715,1,3,6,3,0.009948
0.8682045,0,2,6,7,0.01411764705882353
0.8568936,1,2,6,2,0.38404122807017543
0.8548687,1,3,6,7,0.12
0.85348535,1,3,6,7,0.12
0.8521579,1,3,6,7,0.23
0.85124946,1,3,6,7,0.006857142857142857
0.85119855,1,2,6,7,0.06
0.8504925,1,3,6,3,0.005760869565217391



10% Intervention Strategy
------------------------------------
Customers targeted: 600
Defaults identified: 423
Total defaults in test set: 1327
Default capture rate: 31.88%
Default rate among targeted customers: 70.50%
Expected defaults from random targeting: 132.7
Lift over random targeting: 3.19x

Intervention Capacity Comparison


Intervention Rate,Customers Targeted,Defaults Identified,Default Capture Rate,Targeted Default Rate,Lift vs Random
5%,300,227,0.1710625470987189,0.7566666666666667,3.4212509419743777
10%,600,423,0.31876412961567446,0.705,3.187641296156744
15%,900,563,0.4242652599849284,0.6255555555555555,2.8284350665661897
20%,1200,675,0.5086661642803316,0.5625,2.5433308214016574
25%,1500,779,0.5870384325546345,0.5193333333333333,2.348153730218538



Business Strategy Summary


Intervention Rate,Customers Targeted,Defaults Identified,Default Capture Rate,Targeted Default Rate,Lift vs Random
5%,300,227,17.1%,75.7%,3.42x
10%,600,423,31.9%,70.5%,3.19x
15%,900,563,42.4%,62.6%,2.83x
20%,1200,675,50.9%,56.2%,2.54x
25%,1500,779,58.7%,51.9%,2.35x
